# LiNGAM + Decision Analysis (abalone) on Localstack

Walkthrough: **abalone** from [example-causal-datasets](https://github.com/cmu-phil/example-causal-datasets), **LiNGAM**, then **DA explain**.

## Prepare data

### About this dataset (abalone)

This example uses the **Abalone** dataset from [example-causal-datasets](https://github.com/cmu-phil/example-causal-datasets), originally based on the classic UCI Abalone data.

- Each row is one abalone sample.
- Feature columns (`Length`, `Diam`, `Height`, `Whole`, `Shucked`, `Viscera`, `Shell`) are size/weight measurements.
- `Rings` is used as an age proxy (common approximation: age ~ `Rings + 1.5`).

For DA, target quality depends on whether the current row already satisfies your target. If it already satisfies, action can legitimately be all zeros.


In [ ]:
import tempfile
from pathlib import Path

import pandas as pd

DATA_ROOT = Path.home() / "example-causal-datasets"
ABALONE_TSV = DATA_ROOT / "real/abalone/data/abalone.continuous.txt"

if not ABALONE_TSV.is_file():
    raise FileNotFoundError(
        f"Expected abalone at {ABALONE_TSV}. Clone https://github.com/cmu-phil/example-causal-datasets into {DATA_ROOT}"
    )

_tmp = Path(tempfile.mkdtemp(prefix="abalone_lingam_"))
ABALONE_CSV = _tmp / "abalone.csv"
_abalone_df = pd.read_csv(ABALONE_TSV, sep="\t")
print(_abalone_df.head().to_string())
_abalone_df.to_csv(ABALONE_CSV, index=False)
print("CSV for upload:", ABALONE_CSV)

## Quick EDA for DA target design

Before calling DA, inspect `Rings` distribution and choose:

1. A target threshold that is not trivially satisfied.
2. A row that currently does **not** satisfy the target.

This makes `action` outputs easier to interpret.

In [ ]:
rings = _abalone_df["Rings"]

print("Rows:", len(_abalone_df))
print("Rings quantiles:")
print(rings.quantile([0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).to_string())

print("\nP(Rings > t):")
for t in [8, 10, 12, 14, 16, 18]:
    n = int((rings > t).sum())
    print(f"  > {t}: {n / len(rings):.3f} ({n}/{len(rings)})")

# Set a non-trivial target so we can observe non-zero DA actions.
RINGS_TARGET = 16.0
TARGET_SENSE = ">"

candidate_idx = _abalone_df.index[_abalone_df["Rings"] < RINGS_TARGET].tolist()
if not candidate_idx:
    raise ValueError("No rows found that violate the configured target.")

ROW_INDEX = int(candidate_idx[0])
print(
    f"\nChosen DA setup: Rings {TARGET_SENSE} {RINGS_TARGET}, "
    f"row_index={ROW_INDEX}, current Rings={_abalone_df.loc[ROW_INDEX, 'Rings']}"
)

## API URL and API key

Configure the local API URL and provide an API key to call the endpoints.


In [ ]:
import sys
from pathlib import Path


def _examples_dir_with_helpers() -> Path:
    cwd = Path.cwd().resolve()
    for anchor in [cwd, *cwd.parents[:12]]:
        for rel in (
            Path("MVP/causal_ai_sdk/examples/helpers.py"),
            Path("causal_ai_sdk/examples/helpers.py"),
        ):
            p = anchor / rel
            if p.is_file():
                return p.parent
    raise ImportError(
        "Cannot find MVP/causal_ai_sdk/examples/helpers.py. Open the causal_ai repo (or a parent "
        "folder that contains it) as your workspace so Python can locate examples/helpers.py."
    )


sys.path.insert(0, str(_examples_dir_with_helpers().resolve()))

from helpers import get_api_key_from_env, get_base_url_from_env

BASE_URL = get_base_url_from_env()
if not BASE_URL:
    raise ValueError("No API base URL. Set CAUSAL_AI_BASE_URL.")
API_KEY = get_api_key_from_env()
if not API_KEY:
    raise ValueError("No API key. Set CAUSAL_AI_API_KEY.")
masked_key = f"{API_KEY[:6]}...{API_KEY[-4:]}" if len(API_KEY) > 12 else "[provided]"
print("BASE_URL:", BASE_URL)
print("API_KEY:", masked_key)


#### Client Imports

In [ ]:
import httpx
from causal_ai_sdk import CausalAIClient

### Step 1 — Create a session

In [ ]:
async with CausalAIClient(api_key=API_KEY, base_url=BASE_URL) as client:
    session = await client.kg.init_session()
    SESSION_UUID = session["uuid"]
print("SESSION_UUID:", SESSION_UUID)

### Step 2 — Upload abalone CSV for LiNGAM

The upload info returned is utilized to start LiNGAM.

In [ ]:
async with CausalAIClient(api_key=API_KEY, base_url=BASE_URL) as client:
    UPLOAD_INFO = await client.cd.upload_data_for_lingam(SESSION_UUID, ABALONE_CSV)
print(UPLOAD_INFO.model_dump_json(indent=2))

### Step 3 — Run LiNGAM (submit and wait)

Submit the LiNGAM task, then poll until it finishes.


In [ ]:
async with CausalAIClient(api_key=API_KEY, base_url=BASE_URL) as client:
    cd_task = await client.cd.run_lingam(UPLOAD_INFO, threshold=0.01)
    CD_TASK_ID = cd_task["task_id"]
    print("CD_TASK_ID:", CD_TASK_ID)
    await client.cd.wait_for_task(
        CD_TASK_ID, session_uuid=SESSION_UUID, timeout=600, interval=5
    )
print("LiNGAM completed.")


### Step 4 — Fetch CD result JSON

Download the CD artifact and print a **trimmed JSON preview**; `adjacency_matrices` and `training_data.data` each show only the **first 2 rows**. Full `CD_RESULT` stays in memory for later steps.

**Ensure that you click "View as a scrollable element".**


In [ ]:
import copy
import json

async with CausalAIClient(api_key=API_KEY, base_url=BASE_URL) as client:
    task_result = await client.cd.get_task_result(SESSION_UUID, CD_TASK_ID)
    url = task_result.get("result_url")
    if not url:
        raise RuntimeError("No CD result_url")
    async with httpx.AsyncClient() as http:
        r = await http.get(url)
        r.raise_for_status()
        CD_RESULT = r.json()
if CD_RESULT.get("status") != "succeeded":
    raise RuntimeError(f"CD status: {CD_RESULT.get('status')}")

_cd_preview = copy.deepcopy(CD_RESULT)
for _k in ("uuid", "task_id", "mode", "status", "timestamp"):
    _cd_preview.pop(_k, None)

_cd_preview["adjacency_matrices"] = (_cd_preview.get("adjacency_matrices"))[:2]
_td = _cd_preview["training_data"]
_td["data"] = (_td.get("data"))[:2]

print(json.dumps(_cd_preview, indent=2, default=str))

### Helpers for DA

`_build_one_hot_set` and `_row_to_observation` help us obtain the `current_observation` dict DA expects.

In [ ]:
import math
from typing import Any, Dict

import httpx
from causal_ai_sdk import CausalAIClient


def _build_one_hot_set(cd_result: Dict[str, Any]) -> set:
    names = cd_result.get("feature_names") or []
    categories = (cd_result.get("feature_metadata") or {}).get("feature_categories") or []
    one_hot = {n for n in names if "=" in n}
    for group in categories:
        if isinstance(group, list):
            for idx in group:
                if isinstance(idx, int) and 0 <= idx < len(names):
                    one_hot.add(names[idx])
    return one_hot


def _row_to_observation(cd_result: Dict[str, Any], row_index: int = 0) -> Dict[str, float]:
    names = cd_result.get("feature_names") or []
    rows = (cd_result.get("training_data") or {}).get("data") or []
    if not names or not rows:
        raise ValueError("CD result missing feature_names or training_data.data")
    if row_index < 0 or row_index >= len(rows):
        raise IndexError(f"row_index {row_index} out of range (n={len(rows)})")
    row = rows[row_index]
    if len(row) != len(names):
        raise ValueError("Row length does not match feature_names")
    one_hot = _build_one_hot_set(cd_result)
    obs: Dict[str, float] = {}
    for name, value in zip(names, row):
        if name in one_hot:
            obs[name] = int(value) if value in (0, 1, 0.0, 1.0, False, True) else 0
        else:
            obs[name] = 0.0 if isinstance(value, float) and math.isnan(value) else float(value)
    return obs

### Step 5 — DA explain (submit and wait)

#### Why this target?
- We set `RINGS_TARGET = 16.0` because it is near the high tail (`~95th percentile`) of `Rings`.
- This avoids trivial goals such as `Rings > 4`, which are already satisfied by most rows.
- The selected row (`ROW_INDEX`) is chosen to **violate** the target, so DA has work to do.

#### Why constraints?
- `Rings` is the **target outcome** (age proxy), so we mark it immutable to avoid "directly changing" the outcome variable.
- For a growth-style objective (`Rings > target`), we can require candidate drivers to be `increase_only`.
- In this demo, we keep all measured size/weight features actionable but monotonic (increase-only):
  - `Length`, `Diam`, `Height`, `Whole`, `Shucked`, `Viscera`, `Shell`

Submit DA **explain**, then poll until the task completes.


In [ ]:
current_observation = _row_to_observation(CD_RESULT, row_index=ROW_INDEX)
targets = [{"col": "Rings", "sense": TARGET_SENSE, "threshold": RINGS_TARGET}]

# Constraint design for explainability:
# - Do not intervene directly on the outcome variable (`Rings`).
# - Allow only non-decreasing interventions on measured size/weight features.
constraints = {
    "immutable": ["Rings"],
    "increase_only": [
        "Length",
        "Diam",
        "Height",
        "Whole",
        "Shucked",
        "Viscera",
        "Shell",
    ],
}

cd_ref = {"session_uuid": SESSION_UUID, "task_id": CD_TASK_ID}
print("ROW_INDEX:", ROW_INDEX)
print("Current Observation Rings:", current_observation.get("Rings"))
print("Targets:", targets)
print("Constraints:", constraints)

async with CausalAIClient(api_key=API_KEY, base_url=BASE_URL) as client:
    da_task = await client.da.run_explain(
        session_uuid=SESSION_UUID,
        cd_result_reference=cd_ref,
        current_observation=current_observation,
        targets=targets,
        constraints=constraints,
    )
    DA_TASK_ID = da_task["task_id"]
    print("DA_TASK_ID:", DA_TASK_ID)
    await client.da.wait_for_task(
        SESSION_UUID, DA_TASK_ID, timeout=300, interval=5
    )
print("DA completed.")


### Step 6 — Fetch DA result

Print **`is_solved`** and the recommended feature deltas **`action`**.


In [ ]:
async with CausalAIClient(api_key=API_KEY, base_url=BASE_URL) as client:
    da_meta = await client.da.get_task_result(SESSION_UUID, DA_TASK_ID)
    da_url = da_meta.get("result_url")
    if not da_url:
        raise RuntimeError("No DA result_url")
    async with httpx.AsyncClient() as http:
        r = await http.get(da_url)
        r.raise_for_status()
        da_result = r.json()
inner = da_result.get("result") or {}
print("is_solved:", inner.get("is_solved"))
print("action:", inner.get("action"))

### Step 7 — Cleanup


In [ ]:
async with CausalAIClient(api_key=API_KEY, base_url=BASE_URL) as client:
    if DA_TASK_ID:
        try:
            await client.da.delete_task(SESSION_UUID, DA_TASK_ID)
            print("DA task deleted")
        except Exception as e:
            print("DA delete:", e)
    if CD_TASK_ID:
        try:
            await client.cd.delete_task(SESSION_UUID, CD_TASK_ID)
            print("CD task deleted")
        except Exception as e:
            print("CD delete:", e)
    try:
        await client.kg.delete_session(SESSION_UUID)
        print("Session deleted")
    except Exception as e:
        print("Session delete:", e)